# GTR 직접보상 강화학습
최신 puyo_colab_source.zip을 Colab에 업로드하고 GPU 커널에서 모두 실행하세요.
학습 5,000판 · 1-ply → 최종 평가 250판 · 3-ply 전수 탐색.
결과는 Google Drive의 puyo_runs/gtr_direct 아래 실행별 폴더에 저장됩니다.
처음 실행할 때 resume = None을 사용합니다. 이어서 학습할 때는 정확한 last_resume.pt 경로를 입력하세요.


In [1]:
# 1. GPU와 소스 확인 · 구글 드라이브에 결과 저장
from pathlib import Path
import sys, subprocess, sysconfig, tempfile, zipfile
from datetime import datetime, timezone
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 연결되지 않았습니다. Colab GPU 커널을 선택하세요.")
archive = Path("/content/puyo_colab_source.zip")
if not archive.is_file():
    raise FileNotFoundError("VS Code 탐색기에서 puyo_colab_source.zip → Upload to Colab을 선택하세요.")
drive.mount("/content/drive")
runs = Path("/content/drive/MyDrive/puyo_runs/gtr_direct")
runs.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(), runs)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NVIDIA L4 /content/drive/MyDrive/puyo_runs/gtr_direct


In [2]:
# 2. 이번 런타임에 맞게 게임 엔진 빌드
project = Path(tempfile.mkdtemp(prefix="puyo-", dir="/content"))
with zipfile.ZipFile(archive) as src:
    src.extractall(project)
def run(args):
    # Display live logs and terminate this child if the cell is interrupted.
    child = subprocess.Popen(args, cwd=project, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in child.stdout:
            print(line, end="", flush=True)
        if child.wait():
            raise subprocess.CalledProcessError(child.returncode, args)
    finally:
        if child.poll() is None:
            child.terminate()
            try:
                child.wait(timeout=10)
            except subprocess.TimeoutExpired:
                child.kill()
                child.wait()
run([sys.executable, "-m", "pip", "install", "-q", "pybind11==3.1.0"])
import pybind11
extension = "rl_env/puyo_cpp" + sysconfig.get_config_var("EXT_SUFFIX")
run(["g++", "-std=c++20", "-O2", "-march=native", "-DPEXT", "-shared", "-fPIC",
     "-I" + pybind11.get_include(), "-I" + sysconfig.get_path("include"),
     *sorted(str(p) for p in (project / "core").glob("*.cpp")), "rl_env/bindings.cpp", "-o", extension])
run([sys.executable, "-c", "from rl_env import puyo_cpp; e=puyo_cpp.PuyoEnv(); e.reset(12345); assert e.clone().preview_all()"])
print("게임 엔진 준비 완료")

게임 엔진 준비 완료


In [ ]:
# 3. GTR 직접보상 학습 5,000판 → 마지막에 3수 평가 250판
resume = None  # 재개 시 예: Path("/content/drive/MyDrive/puyo_runs/.../last_resume.pt")
if resume is not None:
    resume = Path(resume)
    assert resume.is_file(), f"재개 파일을 찾을 수 없습니다: {resume}"

destination = runs / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
args = [
    sys.executable, "-u", "puyo_train.py",
    "--device", "cuda",
    "--episodes", "5000",
    "--train-ply", "1",
    "--eval-every", "5000",
    "--eval-episodes", "250",
    "--eval-ply", "3",
    "--eval-seed", "100000",
    "--eval-root-keep", "22",
    "--eval-second-keep", "484",
    "--resume-every", "500",
    "--form-weight", "0.04",
    "--run-dir", str(destination),
]
if resume is not None:
    args.extend(["--resume", str(resume)])
print("저장 위치:", destination, flush=True)
run(args)
print("최종 모델:", destination / "ep_005000.pt")
print("평가 결과:", destination / "evaluations/ep_005000_ply3/summary.json")
